In [13]:
import os
from dotenv import load_dotenv, find_dotenv

In [14]:
load_dotenv(find_dotenv(), override=True)
api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_ENDPOINT")

In [8]:
from openai import OpenAI

client = OpenAI(
    api_key=api_key, 
    base_url=base_url
)

def ask_openai(query):
    stream = client.chat.completions.crate(
        model = "gpt-realtime-mini",
        messages = [
            {
                "role" :"system",
                "content" :"you are a helpful ai assistant"
            },
            {
                "role" :"user",
                "content": query
            }
        ],
        stream = True
    )

    for chunk in stream:
        if chunk.choices[0].delta.content is not None:
            print (chunk.choices[0].delta.content, end='')

In [9]:
ask_openai("What is OpenAI")

AttributeError: 'Completions' object has no attribute 'crate'

In [10]:
pip install websockets

  Using cached websockets-16.0-cp313-cp313-win_amd64.whl.metadata (7.0 kB)
Using cached websockets-16.0-cp313-cp313-win_amd64.whl (178 kB)
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import json
import websockets
from dotenv import load_dotenv

load_dotenv(".env")

API_KEY = os.getenv("OPENAI_API_KEY")
WSS_ENDPOINT = os.getenv("WSS_ENDPOINT")

async def realtime_chat():
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "OpenAI-Beta": "realtime=v1"
    }

    async with websockets.connect(
        WSS_ENDPOINT,
        additional_headers=headers
    ) as ws:

        # ✅ Step 1: send user message (conversation item)
        await ws.send(json.dumps({
            "type": "conversation.item.create",
            "item": {
                "type": "message",
                "role": "user",
                "content": [
                    {"type": "input_text", "text": "What is OpenAI?"}
                ]
            }
        }))

        # ✅ Step 2: trigger response
        await ws.send(json.dumps({
            "type": "response.create"
        }))

        # ✅ Step 3: read streaming output
        while True:
            msg = await ws.recv()
            data = json.loads(msg)

            # Debug (optional)
            # print(data)

            if data.get("type") == "response.output_text.delta":
                print(data.get("delta", ""), end="")

            if data.get("type") == "response.completed":
                break

await realtime_chat()